# 📊 PPTX Analyzer — Lokale PowerPoint-Analyse mit Vision-LLM

Dieses Notebook analysiert PowerPoint-Dateien in vier klar getrennten Schritten:

| Schritt | Was passiert | Ergebnis |
|---------|-------------|----------|
| **0a** | Rohtexte und Bilder extrahieren, **Duplikate erkennen** | `rohtext.md` + `bilder_original/` |
| **0b** | Bilder **formatgerecht** verkleinern (PNG-8 vs JPEG) | `bilder_reduziert/` |
| **1** | LLM-gestützte Bildbeschreibung (nur Unikate) | `bildbeschreibungen.md` + `komplett_extrakt.md` |
| **2** | Thematische Zusammenfassung (Map-Reduce) | `analysierter_extrakt.md` |

**Live-Feedback:** Die Pipeline ist als Generator implementiert — Gradio
aktualisiert die UI nach **jedem einzelnen Bild** und **jedem Chunk**.

**Anti-Truncation:** Wenn das LLM mitten im Satz abbricht, wird
automatisch ein Fortsetzungs-Call gesendet.

**Voraussetzungen:**
- Python ≥ 3.10
- LM Studio mit geladenem Vision-Modell (z.B. Qwen 3.6-27B)
- `pip install gradio==6.8.0 python-pptx requests Pillow`

**Ordnerstruktur nach der Analyse:**
```
📁 Mein_Ordner/
├── Präsentation.pptx                ← Original (wird nicht verändert)
└── Präsentation_workdir/             ← wird automatisch angelegt
    ├── bilder_original/              ← nur Unikate gespeichert
    ├── bilder_reduziert/             ← PNG-8 oder JPEG je nach Bildtyp
    ├── rohtext.md                    ← reiner Text + Tabellen + Notizen
    ├── bildbeschreibungen.md         ← LLM-Bildbeschreibungen (separat)
    ├── …_komplett_extrakt.md         ← Text + Bilder zusammengeführt
    └── …_analysierter_extrakt.md     ← thematische Zusammenfassung
```

## Imports

Wir brauchen nur Standardbibliotheken plus vier Pakete:
- **python-pptx** → PPTX lesen
- **Pillow** → Bilder analysieren und verkleinern
- **requests** → LLM-API ansprechen
- **gradio** → kompakte Web-UI

In [ ]:
import os
import re
import base64
import time
import hashlib
import logging
from pathlib import Path
from collections import defaultdict
from io import BytesIO

from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE
from PIL import Image
import requests
import gradio as gr

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(message)s",
)
log = logging.getLogger("pptx_analyzer")

## Konfiguration

Alle einstellbaren Parameter an **einer** Stelle.

**Zwei-Modell-Strategie:**

| Modell | Aufgabe | Warum |
|--------|---------|-------|
| `MODEL_VISION` (klein, z.B. 9B) | Bildbeschreibungen (Schritt 1) | Schnell, einfache Aufgabe |
| `MODEL_TEXT` (groß, z.B. 27B) | Zusammenfassung (Schritt 2) | Braucht Textverständnis über viele Folien |

Beide laufen in LM Studio parallel über denselben Endpunkt —
im API-Call wird nur der Modellname unterschieden.
Falls nur ein Modell gewünscht: beide auf denselben Namen setzen.

**Dynamische Chunk-Berechnung:** `CHUNK_SIZE` wird aus dem Context Window
von `MODEL_TEXT` berechnet. Bei 30K Tokens: ca. **89 Folien pro Chunk**.

In [ ]:
# -- LM Studio / LLM -------------------------------------------------
API_URL: str = "http://localhost:1234/v1/chat/completions"

#    Zwei-Modell-Strategie:
#    MODEL_VISION  → schnelles, kleines Modell fuer Bildbeschreibungen
#    MODEL_TEXT    → grosses Modell fuer Textanalyse & Zusammenfassung
#
#    Beide laufen in LM Studio parallel (gleicher Endpunkt,
#    unterschiedlicher Modellname im API-Call).
#    Bei 40 GB VRAM: 27B (q4) ≈ 18 GB + 9B (q4) ≈ 6 GB = 24 GB → passt.
#
#    Falls nur EIN Modell gewuenscht: beide auf denselben Namen setzen.
MODEL_VISION: str = "qwen/qwen3.5-9b"
MODEL_TEXT:   str = "qwen/qwen3.6-27b"

LLM_TEMPERATURE    = 0.2
LLM_TIMEOUT        = 180          # Sekunden pro API-Call

# -- Bild-Reduktion --------------------------------------------------
IMAGE_MAX_PX: int  = 768         # laengste Seite in Pixeln
IMAGE_QUALITY: int = 80           # JPEG-Qualitaet (1-95), nur fuer Fotos
GRAPHIC_COLOR_THRESHOLD = 256     # Weniger Farben = Grafik -> PNG-8 behalten

# -- Token-Budgets ----------------------------------------------------
IMAGE_DESCRIBE_TOKENS  = 50       # max Tokens fuer eine Bildbeschreibung
CHUNK_SUMMARY_TOKENS   = 2500     # max Tokens pro Chunk-Zusammenfassung
FINAL_SUMMARY_TOKENS   = 6000     # max Tokens fuer die Gesamtsynthese

# -- Kontextfenster & Chunking ----------------------------------------
#    Bezieht sich auf MODEL_TEXT (das grosse Modell fuer Schritt 2).
CONTEXT_WINDOW_TOKENS: int = 20_000
SYSTEM_PROMPT_TOKENS:  int = 200
ANSWER_RESERVE_TOKENS: int = 3000
TOKENS_PER_SLIDE_EST:  int = 300

CHUNK_SIZE: int = (
    (CONTEXT_WINDOW_TOKENS - SYSTEM_PROMPT_TOKENS - ANSWER_RESERVE_TOKENS)
    // TOKENS_PER_SLIDE_EST
)
log.info(
    "CHUNK_SIZE: %d Folien | Vision: %s | Text: %s",
    CHUNK_SIZE, MODEL_VISION, MODEL_TEXT,
)

## Hilfsfunktionen

### LLM-Aufruf mit Anti-Truncation

`call_llm` kapselt den HTTP-Call an die OpenAI-kompatible API.

**Neu: Automatische Satz-Vervollständigung.** Wenn die Antwort
mitten im Satz abbricht (erkannt über `_is_truncated`), wird
automatisch ein Fortsetzungs-Call gesendet: *"Beende NUR den
letzten Satz."* Das passiert maximal einmal pro Aufruf.

> **Hinweis für Studierende:** Viele lokale LLM-Server (LM Studio, Ollama,
> vLLM, llama.cpp) bieten dieselbe REST-Schnittstelle an. Die Funktion
> funktioniert daher mit jedem dieser Backends — nur `API_URL` und `MODEL`
> anpassen.

In [ ]:
def strip_thinking_tags(text: str) -> str:
    """Entfernt <think>...</think>-Bloecke, die manche Modelle ausgeben."""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def _is_truncated(text: str) -> bool:
    """
    Prueft, ob ein LLM-Antworttext mitten im Satz abgeschnitten wurde.

    Heuristik: Der Text gilt als vollstaendig, wenn er mit einem
    Satzschlusszeichen, einer Markdown-Struktur oder einem
    Aufzaehlungszeichen endet.
    """
    text = text.rstrip()
    if not text:
        return False
    # Satzschlusszeichen, Markdown-Marker, Klammern, Anfuehrungszeichen
    valid_endings = ".!?:;)\u201d\u201c\u00bb|`*_"
    return text[-1] not in valid_endings


def call_llm(
    messages: list[dict],
    max_tokens: int = 500,
    api_url: str = API_URL,
    model: str = MODEL_TEXT,
    auto_continue: bool = True,
) -> str:
    """
    Sendet eine Chat-Completion-Anfrage an die LLM-API.

    Falls die Antwort mitten im Satz abbricht und auto_continue=True,
    wird ein Fortsetzungs-Call gesendet ("Bitte beende den letzten Satz.").
    Das passiert maximal einmal, um Endlosschleifen zu vermeiden.

    Parameters
    ----------
    messages      : OpenAI-kompatibles Nachrichtenformat.
    max_tokens    : Maximale Antwortlaenge.
    api_url       : Endpunkt der API.
    model         : Modellbezeichnung.
    auto_continue : Automatische Fortsetzung bei Abbruch mitten im Satz.
    """
    try:
        resp = requests.post(
            api_url,
            json={
                "model": model,
                "messages": messages,
                "max_tokens": max_tokens,
                "temperature": LLM_TEMPERATURE,
            },
            timeout=LLM_TIMEOUT,
        )
        resp.raise_for_status()
        raw = resp.json()["choices"][0]["message"]["content"]
        text = strip_thinking_tags(raw)

        # -- Fortsetzung bei Abbruch mitten im Satz --
        if auto_continue and _is_truncated(text):
            log.info("Antwort abgeschnitten, sende Fortsetzungs-Call (%s)", model)
            cont_messages = messages + [
                {"role": "assistant", "content": text},
                {"role": "user", "content":
                    "Deine Antwort wurde abgeschnitten. "
                    "Beende NUR den letzten Satz — nichts weiter."},
            ]
            try:
                resp2 = requests.post(
                    api_url,
                    json={
                        "model": model,
                        "messages": cont_messages,
                        "max_tokens": min(max_tokens, 200),
                        "temperature": LLM_TEMPERATURE,
                    },
                    timeout=LLM_TIMEOUT,
                )
                resp2.raise_for_status()
                continuation = strip_thinking_tags(
                    resp2.json()["choices"][0]["message"]["content"]
                )
                if continuation:
                    text = text.rstrip() + " " + continuation.strip()
            except Exception as exc:
                log.warning("Fortsetzungs-Call fehlgeschlagen: %s", exc)

        return text

    except requests.exceptions.Timeout:
        log.warning("Timeout nach %d s (Modell: %s)", LLM_TIMEOUT, model)
        return "[LLM-Fehler: Timeout]"
    except requests.exceptions.ConnectionError:
        return "[LLM-Fehler: Keine Verbindung — laeuft LM Studio?]"
    except Exception as exc:
        log.warning("LLM-Fehler (%s): %s", model, exc)
        return f"[LLM-Fehler: {exc}]"

### PPTX-Extraktionshelfer

PowerPoint-Dateien können Shapes verschachteln (*Gruppierungen*).
`iter_shapes` löst diese rekursiv auf, damit kein Inhalt verloren geht.

`extract_texts_from_shape` liefert sowohl Fließtext als auch Tabellen —
letztere direkt als Markdown-Tabelle formatiert.

In [ ]:
def iter_shapes(shapes):
    """
    Iteriert rekursiv ueber alle Shapes einer Folie.

    Gruppierte Shapes (MSO_SHAPE_TYPE.GROUP) werden aufgeloest,
    sodass auch tief verschachtelte Inhalte erreicht werden.
    """
    for shape in shapes:
        if shape.shape_type == MSO_SHAPE_TYPE.GROUP:
            yield from iter_shapes(shape.shapes)
        else:
            yield shape


def extract_texts_from_shape(shape) -> list[str]:
    """
    Extrahiert lesbaren Text aus einem einzelnen Shape.

    Unterstuetzt:
    - Textrahmen (Titel, Aufzaehlungen, Freitext)
    - Tabellen -> werden als Markdown-Tabelle formatiert
    """
    texts: list[str] = []

    # -- Textrahmen --
    if shape.has_text_frame:
        for para in shape.text_frame.paragraphs:
            line = para.text.strip()
            if line:
                texts.append(line)

    # -- Tabelle -> Markdown --
    if shape.has_table:
        table = shape.table
        rows = list(table.rows)
        if rows:
            header = [cell.text.strip() for cell in rows[0].cells]
            texts.append("| " + " | ".join(header) + " |")
            texts.append("| " + " | ".join(["---"] * len(header)) + " |")
            for row in rows[1:]:
                vals = [cell.text.strip() for cell in row.cells]
                texts.append("| " + " | ".join(vals) + " |")

    return texts


def extract_notes(slide) -> str:
    """Gibt die Sprecher-Notizen einer Folie zurueck (oder leeren String)."""
    if slide.has_notes_slide:
        text = slide.notes_slide.notes_text_frame.text.strip()
        if text:
            return text
    return ""

---
## Schritt 0a — Rohdaten extrahieren (mit Duplikat-Erkennung)

In diesem Schritt passiert **kein LLM-Aufruf**. Wir lesen die PPTX
rein mechanisch aus.

**Neu: Duplikat-Erkennung.** Viele Präsentationen enthalten dasselbe Bild
mehrfach (Logos, wiederkehrende Grafiken). Wir hashen jeden Bild-Blob
per SHA-256 und speichern nur Unikate. Duplikate bekommen im Manifest
einen Verweis (`duplicate_of`) auf das Original.

> **Warum dieser Zwischenschritt?**
> Große Präsentationen (500+ Folien) erzeugen hunderte Bilder.
> Wenn Schritt 1 (LLM) mittendrin abbricht, müssen wir nicht erneut
> extrahieren — die Rohdaten liegen bereits auf der Platte.

In [ ]:
def create_workdir(pptx_path: str) -> dict[str, Path]:
    """
    Legt die Arbeitsordner neben der Original-PPTX an.

    Returns
    -------
    dict  mit Schluesseln 'root', 'img_orig', 'img_small' als Path-Objekte.
    """
    pptx = Path(pptx_path).resolve()
    root      = pptx.parent / f"{pptx.stem}_workdir"
    img_orig  = root / "bilder_original"
    img_small = root / "bilder_reduziert"

    for d in (root, img_orig, img_small):
        d.mkdir(parents=True, exist_ok=True)

    return {"root": root, "img_orig": img_orig, "img_small": img_small}


def blob_hash(data: bytes) -> str:
    """Erzeugt einen kurzen SHA-256-Hash fuer eine Bild-Datei."""
    return hashlib.sha256(data).hexdigest()[:16]


def step_0a_extract(
    pptx_path: str,
    dirs: dict[str, Path],
    progress=None,
) -> tuple[str, list[dict]]:
    """
    Extrahiert Rohtexte und Originalbilder aus der PPTX.

    Duplikate werden per SHA-256-Hash erkannt:
    - Das Bild wird nur einmal gespeichert.
    - Im Manifest erhaelt das Duplikat einen Verweis ('duplicate_of')
      auf den Hash des Originals.

    Returns
    -------
    tuple[str, list[dict]]
        (rohtext_md, bild_manifest)
    """
    prs = Presentation(pptx_path)
    total = len(prs.slides)

    md_lines: list[str] = [
        f"# Rohtext-Extrakt: {Path(pptx_path).stem}",
        f"*{total} Folien — rein mechanisch extrahiert (kein LLM)*\n",
    ]
    bild_manifest: list[dict] = []
    seen_hashes: dict[str, int] = {}   # hash -> Index im Manifest
    dup_count = 0

    for idx, slide in enumerate(prs.slides):
        num = idx + 1
        if progress:
            progress(num / total, desc=f"Schritt 0a: Folie {num}/{total}")

        md_lines.append(f"\n---\n## Folie {num}\n")

        # -- Texte --
        slide_texts: list[str] = []
        for shape in iter_shapes(slide.shapes):
            slide_texts.extend(extract_texts_from_shape(shape))

        if slide_texts:
            md_lines.append("### Text\n")
            md_lines.extend(slide_texts)
            md_lines.append("")

        # -- Notizen --
        notes = extract_notes(slide)
        if notes:
            md_lines.append("### Notizen\n")
            md_lines.append(f"_{notes}_\n")

        # -- Bilder speichern (mit Duplikat-Erkennung) --
        img_counter = 0
        for shape in iter_shapes(slide.shapes):
            if shape.shape_type != MSO_SHAPE_TYPE.PICTURE:
                continue
            try:
                img = shape.image
                img_counter += 1
                h = blob_hash(img.blob)

                ext_map = {
                    "image/png": ".png", "image/jpeg": ".jpg",
                    "image/gif": ".gif", "image/bmp": ".bmp",
                    "image/tiff": ".tiff",
                }
                ext = ext_map.get(img.content_type, ".png")
                filename = f"folie_{num:03d}_bild_{img_counter:02d}{ext}"
                out_path = dirs["img_orig"] / filename

                if h in seen_hashes:
                    # Duplikat: nicht nochmal speichern, nur vermerken
                    dup_count += 1
                    bild_manifest.append({
                        "slide": num,
                        "index": img_counter,
                        "hash": h,
                        "duplicate_of": seen_hashes[h],
                        "path_orig": bild_manifest[seen_hashes[h]]["path_orig"],
                        "content_type": img.content_type or "image/png",
                    })
                else:
                    # Neues Bild: speichern und registrieren
                    out_path.write_bytes(img.blob)
                    seen_hashes[h] = len(bild_manifest)
                    bild_manifest.append({
                        "slide": num,
                        "index": img_counter,
                        "hash": h,
                        "path_orig": out_path,
                        "content_type": img.content_type or "image/png",
                    })

            except Exception as exc:
                log.warning("Folie %d, Bild %d: %s", num, img_counter, exc)

        if img_counter > 0:
            md_lines.append(
                f"*[{img_counter} Bild(er) extrahiert -> bilder_original/]*\n"
            )

    rohtext_md = "\n".join(md_lines)
    rohtext_path = dirs["root"] / "rohtext.md"
    rohtext_path.write_text(rohtext_md, encoding="utf-8")

    unique = len(seen_hashes)
    log.info(
        "Rohtext gespeichert: %s — %d Bilder gesamt, %d unique, %d Duplikate",
        rohtext_path, len(bild_manifest), unique, dup_count,
    )

    return rohtext_md, bild_manifest

---
## Schritt 0b — Bilder formatgerecht verkleinern

Vision-LLMs rechnen pro Bild **Visual Tokens** ab.
Ein 3000 × 2000 px-Foto kann leicht 1000+ Tokens kosten.

### Formatentscheidung: PNG-8 vs. JPEG

### Schritt 1: Alphakanal entfernen (`flatten_alpha`)

Vision-LLMs interpretieren Transparenz als Schwarz — PNG/GIF-Bilder
mit Alphakanal erscheinen dem Modell als „vollständig schwarz".
Deshalb wird **ausnahmslos jedes Bild** zuerst auf einen weißen
Hintergrund compositet, bevor irgendetwas anderes passiert.

### Schritt 2: Formatentscheidung (PNG-8 vs. JPEG)

| Bildtyp | Beispiele | Format | Warum |
|---------|-----------|--------|-------|
| **Grafik** (≤ 256 Farben) | Icons, Häkchen, Diagramme, Logos | **PNG-8** | Verlustfrei, oft *kleiner* als JPEG |
| **Foto** (> 256 Farben) | Screenshots, Fotos, Karten | **JPEG** | Starke Größenreduktion |

Die Erkennung läuft über `is_graphic()` auf dem bereits flachen
RGB-Bild: auf 100×100 px herunterrechnen, Farben zählen.

**Duplikate werden übersprungen** — sie verweisen auf das bereits
verarbeitete Originalbild.

In [ ]:
def flatten_alpha(img: Image.Image, bg_color=(255, 255, 255)) -> Image.Image:
    """
    Entfernt den Alphakanal, indem das Bild auf einen
    einfarbigen Hintergrund (Standard: weiss) compositet wird.

    Muss VOR jeder weiteren Verarbeitung laufen, da Vision-LLMs
    Transparenz als Schwarz interpretieren.
    """
    if img.mode in ("RGBA", "LA", "PA"):
        background = Image.new("RGB", img.size, bg_color)
        # split()[-1] ist der Alphakanal als Maske
        background.paste(img, mask=img.split()[-1])
        return background
    if img.mode == "P":
        # Palettenbild: kann versteckte Transparenz haben
        img = img.convert("RGBA")
        background = Image.new("RGB", img.size, bg_color)
        background.paste(img, mask=img.split()[-1])
        return background
    if img.mode != "RGB":
        return img.convert("RGB")
    return img


def is_graphic(img: Image.Image, threshold: int = GRAPHIC_COLOR_THRESHOLD) -> bool:
    """
    Heuristik: Ist das Bild eine Grafik (wenige Farben) oder ein Foto?

    Wir verkleinern auf 100x100 und zaehlen die eindeutigen Farben.
    Grafiken (Icons, Diagramme, Haken/Kreuze) haben typischerweise
    < 256 Farben.  Fotos haben tausende.

    Parameters
    ----------
    img       : PIL-Image, muss bereits RGB sein (nach flatten_alpha).
    threshold : Farbgrenze. Darunter = Grafik, darueber = Foto.
    """
    sample = img.copy()
    sample.thumbnail((100, 100), Image.NEAREST)

    colors = sample.getcolors(maxcolors=threshold + 1)
    if colors is None:
        return False   # Mehr als threshold Farben -> Foto
    return len(colors) <= threshold


def optimize_graphic_png(img: Image.Image, out_path: Path, max_px: int) -> Path:
    """
    Speichert eine Grafik als optimiertes PNG (Palette).

    Das Bild ist bereits RGB (Alpha wurde in flatten_alpha entfernt).
    """
    w, h = img.size
    if max(w, h) > max_px:
        ratio = max_px / max(w, h)
        img = img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)

    # RGB -> Palette (max 256 Farben, kein Dithering)
    img = img.quantize(colors=256, method=Image.MEDIANCUT, dither=0)

    out_path = out_path.with_suffix(".png")
    img.save(out_path, "PNG", optimize=True)
    return out_path


def optimize_photo_jpeg(img: Image.Image, out_path: Path,
                        max_px: int, quality: int) -> Path:
    """
    Speichert ein Foto als JPEG.

    Das Bild ist bereits RGB (Alpha wurde in flatten_alpha entfernt).
    """
    w, h = img.size
    if max(w, h) > max_px:
        ratio = max_px / max(w, h)
        img = img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)

    out_path = out_path.with_suffix(".jpg")
    img.save(out_path, "JPEG", quality=quality, optimize=True)
    return out_path


def step_0b_resize_images(
    bild_manifest: list[dict],
    dirs: dict[str, Path],
    max_px: int = IMAGE_MAX_PX,
    quality: int = IMAGE_QUALITY,
    progress=None,
) -> list[dict]:
    """
    Verkleinert alle extrahierten Bilder formatgerecht.

    Verarbeitungsreihenfolge pro Bild:
    1. flatten_alpha  — Alphakanal entfernen (IMMER, sonst schwarze Bilder im LLM)
    2. is_graphic     — Farbanalyse auf dem bereits flachen RGB-Bild
    3. optimize_*     — Formatgerechte Kompression

    Duplikate (erkennbar an 'duplicate_of' im Manifest) werden uebersprungen.
    """
    unique_entries = [e for e in bild_manifest if "duplicate_of" not in e]
    total = len(unique_entries)

    if total == 0:
        log.info("Keine Bilder zum Verkleinern.")
        return bild_manifest

    stats = {"graphic_png": 0, "photo_jpg": 0}

    for i, entry in enumerate(unique_entries):
        if progress:
            progress((i + 1) / total, desc=f"Schritt 0b: Bild {i+1}/{total}")

        try:
            img = Image.open(entry["path_orig"])
            stem = entry["path_orig"].stem
            base_path = dirs["img_small"] / stem
            w, h = img.size

            # 1. IMMER zuerst: Alphakanal entfernen
            img = flatten_alpha(img)

            # 2. Formatentscheidung auf dem flachen RGB-Bild
            if is_graphic(img):
                out_path = optimize_graphic_png(img, base_path, max_px)
                entry["format"] = "png"
                stats["graphic_png"] += 1
            else:
                out_path = optimize_photo_jpeg(img, base_path, max_px, quality)
                entry["format"] = "jpg"
                stats["photo_jpg"] += 1

            entry["path_small"] = out_path

            orig_kb = entry["path_orig"].stat().st_size / 1024
            new_kb  = out_path.stat().st_size / 1024
            fmt_tag = "PNG-8" if entry["format"] == "png" else "JPEG"
            log.info(
                "  %s [%s]: %dx%d (%.0f KB -> %.0f KB)",
                stem, fmt_tag, w, h, orig_kb, new_kb,
            )

        except Exception as exc:
            log.warning("Resize fehlgeschlagen: %s — %s",
                        entry["path_orig"].name, exc)
            entry["path_small"] = entry["path_orig"]
            entry["format"] = "orig"

    # Duplikate verlinken
    for entry in bild_manifest:
        if "duplicate_of" in entry:
            orig = bild_manifest[entry["duplicate_of"]]
            entry["path_small"] = orig.get("path_small", orig["path_orig"])
            entry["format"] = orig.get("format", "orig")

    # Gesamtstatistik
    orig_mb = sum(
        e["path_orig"].stat().st_size for e in unique_entries
    ) / 1024**2
    small_mb = sum(
        e["path_small"].stat().st_size for e in unique_entries
        if "path_small" in e
    ) / 1024**2
    factor = orig_mb / small_mb if small_mb > 0 else 0
    dup_count = len(bild_manifest) - total

    log.info(
        "Bildreduktion: %.1f MB -> %.1f MB (Faktor %.1fx) | "
        "%d PNG-8, %d JPEG, %d Duplikate uebersprungen",
        orig_mb, small_mb, factor,
        stats["graphic_png"], stats["photo_jpg"], dup_count,
    )

    return bild_manifest

---
## Schritt 1 — Komplett-Extrakt mit LLM-Bildbeschreibung

Für **jedes reduzierte Unikat-Bild** wird ein LLM-Call ausgeführt —
mit `MODEL_VISION` (dem kleinen, schnellen Modell).
Duplikate erhalten automatisch die Beschreibung des Originals.

Die Bild-Schleife wird **von der Pipeline gesteuert** (nicht von
dieser Funktion), damit nach jedem Bild ein `yield` erfolgt und
der User Live-Feedback in der Gradio-UI sieht.

`describe_image` sendet:
1. Das Bild als Base64 (mit korrektem MIME-Type: `image/png` oder `image/jpeg`)
2. Den umliegenden Folientext als Kontext

`assemble_komplett_extrakt` fügt am Ende alles zusammen — ohne
eigene LLM-Calls.

**Neu: `bildbeschreibungen.md`** — Die Bildbeschreibungen werden
zusätzlich als eigene Datei gespeichert (`save_bildbeschreibungen`).
Damit existieren `rohtext.md` und `bildbeschreibungen.md` als
sauberes Rohdaten-Paar, und Schritt 2 kann mit anderen Parametern
wiederholt werden, ohne Schritt 1 erneut zu durchlaufen.

In [ ]:
def image_to_base64(path: Path) -> str:
    """Liest eine Bilddatei und gibt den Base64-String zurueck."""
    return base64.b64encode(path.read_bytes()).decode()


def mime_for_format(fmt: str) -> str:
    """Gibt den MIME-Type fuer das reduzierte Bildformat zurueck."""
    return "image/png" if fmt == "png" else "image/jpeg"


def describe_image(
    image_path: Path,
    image_format: str = "jpg",
    slide_context: str = "",
    api_url: str = API_URL,
    model: str = MODEL_VISION,
) -> str:
    """
    Laesst das Vision-LLM ein einzelnes Bild beschreiben.

    Nutzt standardmaessig MODEL_VISION (das kleinere, schnellere Modell).
    """
    b64 = image_to_base64(image_path)
    mime = mime_for_format(image_format)

    prompt = (
        "Beschreibe dieses Bild praezise auf Deutsch. "
        "Was wird dargestellt? Welche Informationen sind enthalten? "
        "Beende deine Antwort mit einem vollstaendigen Satz."
    )
    if slide_context:
        prompt += f"\n\nKontext der Folie:\n{slide_context[:500]}"

    messages = [{
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": {"url": f"data:{mime};base64,{b64}"},
            },
            {"type": "text", "text": prompt},
        ],
    }]
    return call_llm(messages, max_tokens=IMAGE_DESCRIBE_TOKENS,
                    api_url=api_url, model=model)


def parse_slide_texts(rohtext_md: str) -> dict[int, str]:
    """
    Parst den Rohtext und gibt pro Folie den Text als Kontext zurueck.
    """
    slide_texts: dict[int, str] = {}
    current_slide = 0
    current_lines: list[str] = []
    for line in rohtext_md.split("\n"):
        m = re.match(r"^## Folie (\d+)", line)
        if m:
            if current_slide > 0:
                slide_texts[current_slide] = "\n".join(current_lines)
            current_slide = int(m.group(1))
            current_lines = []
        else:
            current_lines.append(line)
    if current_slide > 0:
        slide_texts[current_slide] = "\n".join(current_lines)
    return slide_texts


def save_bildbeschreibungen(
    bild_manifest: list[dict],
    hash_to_desc: dict[str, str],
    dirs: dict[str, Path],
    pptx_path: str,
    model_vision: str = MODEL_VISION,
) -> str:
    """
    Speichert alle Bildbeschreibungen als eigene Markdown-Datei.

    Getrennt vom Komplett-Extrakt, damit:
    - Die Qualitaet isoliert geprueft werden kann
    - Schritt 2 mit anderen Parametern wiederholbar ist
    - Die Rohdaten (rohtext.md + bildbeschreibungen.md) als Paar
      die komplette Grundlage bilden

    Returns
    -------
    str   Der Markdown-Inhalt der Bildbeschreibungen.
    """
    stem = Path(pptx_path).stem
    unique_count = len(hash_to_desc)
    dup_count = sum(1 for e in bild_manifest if "duplicate_of" in e)

    lines: list[str] = [
        f"# Bildbeschreibungen: {stem}",
        f"*{unique_count} Bilder beschrieben, {dup_count} Duplikate | "
        f"Vision-Modell: {model_vision}*\n",
    ]

    current_slide = None
    for entry in bild_manifest:
        # Neue Folien-Ueberschrift bei Folienwechsel
        if entry["slide"] != current_slide:
            current_slide = entry["slide"]
            lines.append(f"\n## Folie {current_slide}\n")

        h = entry["hash"]
        desc = hash_to_desc.get(h, "[Beschreibung nicht verfuegbar]")
        is_dup = "duplicate_of" in entry
        dup_tag = " *(Duplikat)*" if is_dup else ""
        lines.append(f"**Bild {entry['index']}:** {desc}{dup_tag}\n")

    md_text = "\n".join(lines)

    out_path = dirs["root"] / "bildbeschreibungen.md"
    out_path.write_text(md_text, encoding="utf-8")
    log.info("Bildbeschreibungen gespeichert: %s", out_path)

    return md_text


def assemble_komplett_extrakt(
    pptx_path: str,
    rohtext_md: str,
    bild_manifest: list[dict],
    hash_to_desc: dict[str, str],
    model_vision: str = MODEL_VISION,
) -> str:
    """
    Baut den Komplett-Extrakt zusammen:
    Rohtext + bereits berechnete Bildbeschreibungen in einem Dokument.

    Diese Funktion macht KEINEN LLM-Call — die Beschreibungen
    kommen fertig aus hash_to_desc.
    """
    stem = Path(pptx_path).stem
    unique_count = len(hash_to_desc)
    dup_count = sum(1 for e in bild_manifest if "duplicate_of" in e)

    bild_desc: dict[int, list[str]] = defaultdict(list)
    for entry in bild_manifest:
        h = entry["hash"]
        desc = hash_to_desc.get(h, "[Beschreibung nicht verfuegbar]")
        if "duplicate_of" in entry:
            desc = f"{desc} *(Duplikat)*"
        bild_desc[entry["slide"]].append(desc)

    output_lines: list[str] = [
        f"# Komplett-Extrakt: {stem}",
        f"*Quelle: {Path(pptx_path).name} | {unique_count} Bilder analysiert"
        f" ({dup_count} Duplikate) | Vision: {model_vision}*\n",
    ]

    current_slide = 0
    for line in rohtext_md.split("\n"):
        if line.startswith("# Rohtext-Extrakt:"):
            continue
        if line.startswith("*") and "mechanisch" in line:
            continue

        m_slide = re.match(r"^## Folie (\d+)", line)
        if m_slide:
            current_slide = int(m_slide.group(1))

        m_img = re.match(r"^\*\[\d+ Bild\(er\) extrahiert", line)
        if m_img:
            descs = bild_desc.get(current_slide, [])
            if descs:
                output_lines.append("\n### Bildbeschreibungen\n")
                for j, d in enumerate(descs, 1):
                    output_lines.append(f"**Bild {j}:** {d}\n")
            continue

        output_lines.append(line)

    return "\n".join(output_lines)

---
## Schritt 2 — Analyse & Zusammenfassung (Map-Reduce)

Bei hunderten Folien passt der Komplett-Extrakt nicht in ein einziges
Context Window. Daher nutzen wir eine **Map-Reduce-Strategie**:

```
                    Komplett-Extrakt
     (Chunk 1)       (Chunk 2)       (Chunk 3)    …
         │               │               │
         ▼               ▼               ▼
   Chunk-Summary   Chunk-Summary   Chunk-Summary     ← MAP
         │               │               │
         └───────┬───────┘
                 ▼
        Gesamtzusammenfassung                        ← REDUCE
```

**CHUNK_SIZE wird dynamisch berechnet** aus dem Context Window
(siehe Konfiguration). Bei 30K Tokens: ca. 89 Folien pro Chunk.

Auch hier steuert die Pipeline die Schleife, damit zwischen
den MAP-Chunks ge-yieldet werden kann. Alle Calls in Schritt 2
gehen an `MODEL_TEXT` (das große Modell).

In [ ]:
SYSTEM_CHUNK = (
    "Du bist ein Experte fuer Praesentationsanalyse. "
    "Erstelle eine strukturierte, verstaendliche Zusammenfassung auf Deutsch. "
    "Beruecksichtige Texte UND Bildbeschreibungen gleichermassen. "
    "Fasse thematisch zusammen, nicht folienweise. "
    "Nutze Markdown-Formatierung. "
    "WICHTIG: Beende deine Antwort immer mit einem vollstaendigen Satz."
)

SYSTEM_FINAL = (
    "Du bist ein Experte fuer Praesentationsanalyse. "
    "Erstelle aus den folgenden Teilzusammenfassungen eine kohaerente "
    "Gesamtzusammenfassung auf Deutsch. Strukturiere nach Themen, "
    "vermeide Redundanzen, behalte die wichtigsten Details bei. "
    "Nutze Markdown-Formatierung. "
    "WICHTIG: Beende deine Antwort immer mit einem vollstaendigen Satz."
)


def split_into_chunks(komplett_md: str) -> list[str]:
    """
    Teilt den Komplett-Extrakt an Folien-Ueberschriften in Chunks.
    Jeder Chunk enthaelt CHUNK_SIZE Folien.
    """
    parts = re.split(r"(?=\n---\n## Folie \d+)", komplett_md)
    slide_sections = parts[1:] if len(parts) > 1 else parts

    chunks: list[str] = []
    for ci in range(0, len(slide_sections), CHUNK_SIZE):
        chunk_text = "\n".join(slide_sections[ci : ci + CHUNK_SIZE])
        chunks.append(chunk_text)

    return chunks


def summarize_chunk(
    chunk_text: str,
    api_url: str = API_URL,
    model: str = MODEL_TEXT,
) -> str:
    """Fasst einen einzelnen Chunk zusammen. Nutzt MODEL_TEXT (grosses Modell)."""
    messages = [
        {"role": "system", "content": SYSTEM_CHUNK},
        {"role": "user", "content":
            "Analysiere und fasse diesen Abschnitt zusammen:\n\n"
            + chunk_text},
    ]
    return call_llm(messages, max_tokens=CHUNK_SUMMARY_TOKENS,
                    api_url=api_url, model=model)


def synthesize_summaries(
    chunk_summaries: list[str],
    api_url: str = API_URL,
    model: str = MODEL_TEXT,
) -> str:
    """REDUCE-Schritt: Verdichtet Teilzusammenfassungen zur Gesamtsynthese."""
    if len(chunk_summaries) == 1:
        return chunk_summaries[0]
    if len(chunk_summaries) == 0:
        return "*Keine auswertbaren Inhalte gefunden.*"

    combined = "\n\n---\n\n".join(
        f"### Teilabschnitt {i}\n\n{s}"
        for i, s in enumerate(chunk_summaries, 1)
    )
    messages = [
        {"role": "system", "content": SYSTEM_FINAL},
        {"role": "user", "content":
            "Erstelle eine Gesamtzusammenfassung:\n\n" + combined},
    ]
    return call_llm(messages, max_tokens=FINAL_SUMMARY_TOKENS,
                    api_url=api_url, model=model)


def assemble_analysierter_extrakt(
    pptx_path: str,
    final_summary: str,
    chunk_summaries: list[str],
    n_slides: int,
    dirs: dict[str, Path],
    model: str = MODEL_TEXT,
) -> str:
    """Baut das Markdown fuer den analysierten Extrakt zusammen und speichert es."""
    stem = Path(pptx_path).stem
    total_chunks = len(chunk_summaries)

    md_lines = [
        f"# Analysierter Extrakt: {stem}\n",
        f"*Text-Modell: {model} | {n_slides} Folien "
        f"in {total_chunks} Chunks (je {CHUNK_SIZE} Folien)*\n",
        final_summary,
    ]

    if total_chunks > 1:
        md_lines.append("\n\n---\n## Detailanalysen pro Abschnitt\n")
        for i, s in enumerate(chunk_summaries, 1):
            start = (i - 1) * CHUNK_SIZE + 1
            end = min(i * CHUNK_SIZE, n_slides)
            md_lines.append(f"\n### Folien {start} bis {end}\n\n{s}\n")

    result_md = "\n".join(md_lines)

    out_path = dirs["root"] / f"{stem}_analysierter_extrakt.md"
    out_path.write_text(result_md, encoding="utf-8")
    log.info("Analysierter Extrakt gespeichert: %s", out_path)

    return result_md

---
## Orchestrierung — Die Generator-Pipeline

`run_pipeline` ist ein **Python-Generator** mit Zwei-Modell-Strategie:

- **Schritt 1** → `MODEL_VISION` (schnell, für Bildbeschreibungen)
- **Schritt 2** → `MODEL_TEXT` (stark, für Zusammenfassung)

Beide Modelle werden vorab einzeln getestet. Nach jedem Bild und
jedem Chunk wird ge-yieldet — inklusive **Zeitschätzung** (ETA)
basierend auf der bisherigen Verarbeitungsgeschwindigkeit.

```
yield "Schritt 1 [qwen3.5-9b] Bild 3/186 | ~4:20 verbleibend"
yield "Schritt 2 [qwen3.6-27b] MAP Chunk 1/2"
yield "Fertig!", zusammenfassung
```

In [ ]:
def run_pipeline(
    pptx_path: str,
    api_url: str = API_URL,
    model_vision: str = MODEL_VISION,
    model_text: str = MODEL_TEXT,
):
    """
    Generator-Pipeline mit Zwei-Modell-Strategie.

    - MODEL_VISION (klein/schnell) → Schritt 1 (Bildbeschreibungen)
    - MODEL_TEXT   (gross/stark)   → Schritt 2 (Zusammenfassung)

    Ergebnisse in der Workdir:
    - rohtext.md              ← rein mechanisch (Schritt 0a)
    - bildbeschreibungen.md   ← LLM-Beschreibungen separat (Schritt 1)
    - komplett_extrakt.md     ← beides zusammengefuegt (Schritt 1)
    - analysierter_extrakt.md ← Zusammenfassung (Schritt 2)

    Yields
    ------
    tuple[str, str]   (status_markdown, vorschau_text)
    """
    pptx_path = pptx_path.strip().strip('"').strip("'")

    # -- Validierung --
    if not pptx_path:
        yield "Bitte einen Dateipfad eingeben.", ""
        return
    if not os.path.isfile(pptx_path):
        yield f"Datei nicht gefunden:\n`{pptx_path}`", ""
        return
    if not pptx_path.lower().endswith(".pptx"):
        yield "Nur `.pptx`-Dateien werden unterstuetzt.", ""
        return

    # -- Verbindungstests --
    yield "Teste LLM-Verbindungen ...", ""

    test_v = call_llm(
        [{"role": "user", "content": "Antworte nur mit OK."}],
        max_tokens=10, api_url=api_url, model=model_vision,
        auto_continue=False,
    )
    if test_v.startswith("[LLM-Fehler"):
        yield (
            f"Vision-Modell nicht erreichbar.\n\n"
            f"- **API:** `{api_url}`\n"
            f"- **Modell:** `{model_vision}`\n\n{test_v}"
        ), ""
        return

    if model_text != model_vision:
        test_t = call_llm(
            [{"role": "user", "content": "Antworte nur mit OK."}],
            max_tokens=10, api_url=api_url, model=model_text,
            auto_continue=False,
        )
        if test_t.startswith("[LLM-Fehler"):
            yield (
                f"Text-Modell nicht erreichbar.\n\n"
                f"- **API:** `{api_url}`\n"
                f"- **Modell:** `{model_text}`\n\n{test_t}"
            ), ""
            return

    same = model_vision == model_text
    model_info = (
        f"`{model_vision}`" if same
        else f"Vision: `{model_vision}` | Text: `{model_text}`"
    )
    yield f"LLM verbunden ({model_info}). Starte Analyse ...", ""
    t0 = time.time()
    stem = Path(pptx_path).stem

    # ═══════════════════════════════════════════════════
    # SCHRITT 0a
    # ═══════════════════════════════════════════════════
    yield "**Schritt 0a** — Extrahiere Texte und Bilder ...", ""
    dirs = create_workdir(pptx_path)
    rohtext_md, bild_manifest = step_0a_extract(pptx_path, dirs)

    unique_entries = [e for e in bild_manifest if "duplicate_of" not in e]
    dup_count = len(bild_manifest) - len(unique_entries)

    yield (
        f"**Schritt 0a** — {len(bild_manifest)} Bilder "
        f"({len(unique_entries)} unique, {dup_count} Duplikate)\n\n"
        f"**Schritt 0b** — Verkleinere Bilder ..."
    ), ""

    # ═══════════════════════════════════════════════════
    # SCHRITT 0b
    # ═══════════════════════════════════════════════════
    bild_manifest = step_0b_resize_images(bild_manifest, dirs)

    n_png = sum(1 for e in unique_entries if e.get("format") == "png")
    n_jpg = sum(1 for e in unique_entries if e.get("format") == "jpg")

    # ═══════════════════════════════════════════════════
    # SCHRITT 1 — Bildbeschreibungen (Vision-Modell)
    # ═══════════════════════════════════════════════════
    slide_texts = parse_slide_texts(rohtext_md)
    hash_to_desc: dict[str, str] = {}
    total_unique = len(unique_entries)
    beschreibungen_log: list[str] = []

    for i, entry in enumerate(unique_entries):
        elapsed_now = time.time() - t0
        if i > 0:
            per_img = elapsed_now / i
            remaining = per_img * (total_unique - i)
            m_rem, s_rem = divmod(int(remaining), 60)
            eta = f" | ~{m_rem}:{s_rem:02d} verbleibend"
        else:
            eta = ""

        status = (
            f"**Schritt 1** [`{model_vision}`] — "
            f"Bild **{i+1}** / {total_unique} "
            f"(Folie {entry['slide']}, "
            f"{'PNG' if entry.get('format')=='png' else 'JPEG'})"
            f"{eta}"
        )
        preview = "\n".join(beschreibungen_log[-8:])
        if len(beschreibungen_log) > 8:
            preview = f"*... {len(beschreibungen_log)-8} weitere ...*\n\n" + preview

        yield status, preview

        context = slide_texts.get(entry["slide"], "")
        desc = describe_image(
            entry["path_small"],
            image_format=entry.get("format", "jpg"),
            slide_context=context,
            api_url=api_url, model=model_vision,
        )
        hash_to_desc[entry["hash"]] = desc
        beschreibungen_log.append(
            f"**Folie {entry['slide']}, Bild {entry['index']}:** {desc[:120]}..."
            if len(desc) > 120 else
            f"**Folie {entry['slide']}, Bild {entry['index']}:** {desc}"
        )

    # Bildbeschreibungen separat speichern
    save_bildbeschreibungen(
        bild_manifest, hash_to_desc, dirs, pptx_path,
        model_vision=model_vision,
    )

    # Komplett-Extrakt zusammenbauen (Text + Bilder vereint)
    komplett_md = assemble_komplett_extrakt(
        pptx_path, rohtext_md, bild_manifest, hash_to_desc,
        model_vision=model_vision,
    )
    out_komplett = dirs["root"] / f"{stem}_komplett_extrakt.md"
    out_komplett.write_text(komplett_md, encoding="utf-8")

    elapsed_s1 = time.time() - t0
    m1, s1 = divmod(int(elapsed_s1), 60)

    # ═══════════════════════════════════════════════════
    # SCHRITT 2 — Map-Reduce (Text-Modell)
    # ═══════════════════════════════════════════════════
    chunks = split_into_chunks(komplett_md)
    total_chunks = len(chunks)
    chunk_summaries: list[str] = []

    for ci, chunk_text in enumerate(chunks):
        yield (
            f"**Schritt 2 MAP** [`{model_text}`] — "
            f"Chunk **{ci+1}** / {total_chunks} "
            f"(Schritt 1 dauerte {m1}:{s1:02d} min)"
        ), "\n\n---\n\n".join(chunk_summaries[-3:]) if chunk_summaries else ""

        summary = summarize_chunk(chunk_text, api_url=api_url, model=model_text)
        chunk_summaries.append(summary)

    if total_chunks > 1:
        yield (
            f"**Schritt 2 REDUCE** [`{model_text}`] — "
            f"Verdichte {total_chunks} Teilzusammenfassungen ..."
        ), "\n\n---\n\n".join(chunk_summaries[-3:])

    final_summary = synthesize_summaries(chunk_summaries,
                                         api_url=api_url, model=model_text)

    n_slides = len(re.findall(r"## Folie \d+", komplett_md))
    summary_md = assemble_analysierter_extrakt(
        pptx_path, final_summary, chunk_summaries, n_slides, dirs,
        model=model_text,
    )

    # ═══════════════════════════════════════════════════
    # FERTIG
    # ═══════════════════════════════════════════════════
    elapsed = time.time() - t0
    mins, secs = divmod(int(elapsed), 60)
    p_root = dirs["root"]

    final_status = (
        f"### Analyse abgeschlossen ({mins}:{secs:02d} min)\n\n"
        f"| | |\n|---|---|\n"
        f"| **Vision-Modell** | `{model_vision}` |\n"
        f"| **Text-Modell** | `{model_text}` |\n"
        f"| Rohtext | `{p_root / 'rohtext.md'}` |\n"
        f"| Bildbeschreibungen | `{p_root / 'bildbeschreibungen.md'}` |\n"
        f"| Bilder (original) | `{dirs['img_orig']}/` ({len(unique_entries)} unique) |\n"
        f"| Bilder (reduziert) | `{dirs['img_small']}/` ({n_png} PNG, {n_jpg} JPG) |\n"
        f"| Komplett-Extrakt | `{out_komplett}` |\n"
        f"| Analysierter Extrakt | `{p_root / (stem + '_analysierter_extrakt.md')}` |\n"
        f"\n*{dup_count} Bild-Duplikat(e) erkannt und uebersprungen.*"
    )
    yield final_status, summary_md

---
## Gradio-UI

Eine kompakte Web-Oberfläche mit **Live-Status**.
Da `run_pipeline` ein Generator ist, aktualisiert Gradio
beide Output-Felder bei jedem `yield` automatisch.

> **Alternativ** kann `run_pipeline(...)` auch direkt in einer
> Notebook-Zelle konsumiert werden — z.B. für Batch-Verarbeitung.

In [ ]:
def build_ui() -> gr.Blocks:
    with gr.Blocks(title="PPTX Analyzer", theme=gr.themes.Soft()) as app:
        gr.Markdown(
            "## PPTX Analyzer\n"
            "PowerPoint lokal analysieren und zusammenfassen mit Vision-LLM."
        )

        with gr.Row():
            with gr.Column(scale=3):
                pptx_input = gr.Textbox(
                    label="Pfad zur PowerPoint-Datei",
                    placeholder=r"C:\Dokumente\Deck.pptx  oder  /home/user/deck.pptx",
                    lines=1,
                    info="Vollstaendigen Pfad eingeben oder einfuegen",
                )
            with gr.Column(scale=1):
                api_input = gr.Textbox(
                    label="LM Studio API",
                    value=API_URL,
                    lines=1,
                )
                vision_input = gr.Textbox(
                    label="Vision-Modell (Bilder)",
                    value=MODEL_VISION,
                    lines=1,
                    info="Schnelles Modell fuer Bildbeschreibungen",
                )
                text_input = gr.Textbox(
                    label="Text-Modell (Zusammenfassung)",
                    value=MODEL_TEXT,
                    lines=1,
                    info="Grosses Modell fuer Schritt 2",
                )

        start_btn = gr.Button("Analyse starten", variant="primary", size="lg")
        status_out = gr.Markdown(label="Status")

        with gr.Accordion("Vorschau: Analysierter Extrakt", open=False):
            preview_out = gr.Markdown()

        start_btn.click(
            fn=run_pipeline,
            inputs=[pptx_input, api_input, vision_input, text_input],
            outputs=[status_out, preview_out],
        )

    return app


app = build_ui()
app.launch(inbrowser=True)

---
## Bonus: Batch-Verarbeitung (ohne UI)

Da `run_pipeline` ein Generator ist, muss er durchiteriert werden.
Das letzte `yield`-Ergebnis enthält den finalen Status.

In [ ]:
# Beispiel: mehrere PPTX-Dateien nacheinander verarbeiten
# (Pfade anpassen und Zelle ausfuehren)
#
# Da run_pipeline ein Generator ist, muessen wir ihn durchiterieren.
# Das letzte yield-Ergebnis enthaelt den finalen Status.

DATEIEN = [
    # r"C:\Pfad\zur\ersten.pptx",
    # r"C:\Pfad\zur\zweiten.pptx",
]

for datei in DATEIEN:
    print(f"\n{'='*60}")
    print(f"Verarbeite: {datei}")
    print(f"{'='*60}")

    status = ""
    for status, preview in run_pipeline(datei):
        print(f"  {status[:80]}...")

    print(f"\n{status}")